# 🎤 VoxCPM2 Inference on Colab

**OpenBMB/VoxCPM2** — Zero-shot TTS (Text-to-Speech)

⚡ **Önce şu adımları yap:**
1. `Runtime → Change runtime type → T4 GPU` seç
2. Aşağıdaki tüm hücreleri sırayla çalıştır

⏱️ Toplam süre: ~5-8 dk (sadece ilk seferde)

---
**📦 İki kurulum yöntemi:**
- **A (önerilen):** GitHub'dan clone + pip install (evrensel, her zaman çalışır)
- **B (hızlı):** Pre-built wheel upload (WSL'de `build_wheels.sh` ile üret)

In [ ]:
# === 0. GPU KONTROL ===
import torch
import sys

print(f"Python: {sys.version}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {vram:.1f} GB")
    assert vram >= 14, f"T4 gerekli! Mevcut: {vram:.1f} GB"

In [ ]:
# === 1. BAĞIMLILIKLARI KUR ===
# NOT: flash-attn derleneceği için ~3-4 dk sürebilir. Sabret.

import os, subprocess, sys

def pip_install(pkg, *args):
    cmd = [sys.executable, "-m", "pip", "install", pkg] + list(args) + ["-q"]
    print(f"  ⏳ {pkg.split('==')[0]}...", end=" ")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        print("✅")
    else:
        print(f"❌\n{result.stderr[-300:]}")
    return result.returncode

# 1a. Torch 2.5.1 (2.6.x desteklenmez!)
pip_install("torch==2.5.1 torchaudio==2.5.1",
            "--index-url", "https://download.pytorch.org/whl/cu121")

# 1b. flash-attn (en kritik)
pip_install("flash-attn", "--no-build-isolation", "--no-cache-dir")

# 1c. Diğer bağımlılıklar
pip_install("transformers>=4.51.0 soundfile librosa numpy tqdm xxhash pydantic torchcodec huggingface_hub psutil")

print("\n✅ Tüm bağımlılıklar kuruldu")

In [ ]:
# === 2. KENDİ PAKETİMİZİ KUR ===
# YÖNTEM A (önerilen): GitHub'dan clone

import os

REPO_URL = "https://github.com/kstbhdr/nanovllm-voxcpm-official"
WORK_DIR = "/content/nanovllm-voxcpm-official"

if not os.path.exists(WORK_DIR):
    print("⏳ Repo klonlanıyor...")
    !git clone {REPO_URL} {WORK_DIR} 2>&1 | tail -3
else:
    print("ℹ️  Repo zaten var, güncelleniyor...")
    %cd {WORK_DIR}
    !git pull 2>&1 | tail -3

%cd {WORK_DIR}
!pip install -e . --no-deps -q

from nanovllm_voxcpm import VoxCPM
print(f"✅ VoxCPM paketi hazır (v{VoxCPM.__module__})")

In [ ]:
# === 3. MODELİ İNDİR ===
from huggingface_hub import snapshot_download
import os, json

print("⏳ openbmb/VoxCPM2 indiriliyor (~2.5 GB)...")
model_path = snapshot_download(repo_id="openbmb/VoxCPM2")

# Config kontrol
with open(os.path.join(model_path, "config.json")) as f:
    config = json.load(f)
arch = config.get("architecture")
assert arch == "voxcpm2", f"Beklenen: voxcpm2, Alınan: {arch}"

safetensors = [f for f in os.listdir(model_path) if f.endswith(".safetensors")]
vae_ok = os.path.exists(os.path.join(model_path, "audiovae.pth"))

print(f"  📦 Architecture: {arch}")
print(f"  📦 Safetensors: {len(safetensors)} dosya")
print(f"  🎵 audiovae.pth: {'✅' if vae_ok else '❌'}")
assert vae_ok, "audiovae.pth bulunamadı!"
print(f"\n✅ Model hazır: {model_path}")

In [ ]:
# === 4. INFERENCE ÇALIŞTIR ===
import asyncio
import numpy as np
import time
from IPython.display import Audio, display
from nanovllm_voxcpm import VoxCPM


async def main():
    print("⏳ Model yükleniyor (ilk seferde ~1-2 dk)...")

    server = VoxCPM.from_pretrained(
        model=model_path,
        max_num_batched_tokens=4096,
        max_num_seqs=2,
        max_model_len=2048,
        gpu_memory_utilization=0.85,
        enforce_eager=True,       # T4'te compile kararlı değil
        devices=[0],
    )
    await server.wait_for_ready()
    print("✅ Model hazır!")

    model_info = await server.get_model_info()
    sample_rate = int(model_info["sample_rate"])
    print(f"🔊 Sample rate: {sample_rate} Hz")

    # --- TÜRKÇE METİN ---
    text = "Merhaba dünya! Bugün hava çok güzel, nasılsınız?"
    print(f"\n📝 Metin: {text}")
    print("🔊 Ses sentezleniyor...")

    buf = []
    start_time = time.time()
    async for data in server.generate(
        target_text=text,
        cfg_value=2.0,
        temperature=1.0,
    ):
        buf.append(data)

    wav = np.concatenate(buf, axis=0)
    elapsed = time.time() - start_time
    wav_duration = wav.shape[0] / sample_rate

    print(f"\n⏱️  Süre: {elapsed:.2f}s")
    print(f"🎵 Ses: {wav_duration:.2f}s")
    print(f"📊 RTF: {elapsed / wav_duration:.3f}")

    # Colab'da oynat
    display(Audio(wav, rate=sample_rate))

    await server.stop()
    print("\n✅ Tamamlandı!")


await main()

In [ ]:
# === 5. (OPSİYONEL) SESİ İNDİR ===
import soundfile as sf
from google.colab import files

OUT_FILE = "voxcpm2_output.wav"
sf.write(OUT_FILE, wav, sample_rate)
print(f"💾 Kaydedildi: {OUT_FILE} ({len(wav)} samples)")

# Bilgisayarına indir
files.download(OUT_FILE)
print("✅ İndirme başladı!")

In [ ]:
# === 6. (OPSİYONEL) GELİŞMİŞ KULLANIM ===
# Farklı metin, farklı parametreler

texts = [
    "Merhaba! Ben VoxCPM2, yapay zeka ses sentezleyiciyim.",
    "Bugün hava sıcaklığı 25 derece, nem oranı yüzde 60.",
    "Bu bir test konuşmasıdır. Ses kalitesini değerlendiriyoruz.",
]

for i, text in enumerate(texts):
    print(f"\n--- Örnek {i+1} ---")
    print(f"📝 {text}")

    buf = []
    async for data in server.generate(
        target_text=text,
        cfg_value=2.0,
        temperature=0.9,  # düşük sıcaklık = daha tutarlı
    ):
        buf.append(data)

    wav_i = np.concatenate(buf, axis=0)
    sf.write(f"ornek_{i+1}.wav", wav_i, sample_rate)
    print(f"💾 ornek_{i+1}.wav kaydedildi")
    display(Audio(wav_i, rate=sample_rate))

## 🛠️ Sorun Giderme

### 1. flash-attn kurulum hatası
```
!pip uninstall flash-attn -y
!pip install flash-attn==2.7.4 --no-build-isolation --no-cache-dir
```
Hala hata alıyorsan:
```
!pip install ninja setuptools wheel -q
!MAX_JOBS=2 pip install flash-attn --no-build-isolation
```

### 2. CUDA out of memory (OOM)
T4 16GB'da nadir olur. Yine de olursa:
```python
max_num_batched_tokens=2048
max_num_seqs=1
max_model_len=1024
gpu_memory_utilization=0.80
```

### 3. server process exited early
VRAM yetersiz. Parametreleri küçültüp tekrar dene.
Hücre 4'teki hücreyi tekrar çalıştır (Runtime → Run selection).

### 4. WSL'de (RTX 4060 8GB) çalıştırma
Başarıyla test edilmiştir:
- `enforce_eager=True`
- `max_model_len=2048`
- `max_num_seqs=1`
- `gpu_memory_utilization=0.90`
- `python voxcpm2_run.py`

### 5. Transformer versiyon hatası
```
!pip install transformers>=4.51.0 -q
```